In [1]:
import pandas as pd
import numpy as np

In [2]:
cf = pd.read_csv("../top100_hybrid_candidates.csv")

hybrid = pd.read_csv("../final_hybrid_top10.csv")

ratings = pd.read_csv("../ml-25m/ratings.csv")

In [3]:
sample_user = 99476

user_ratings = ratings[
    ratings["userId"] == sample_user
]

user_ratings.head()

,userId,movieId,rating,timestamp
15347550,99476,6,4.0,1467897620
15347551,99476,16,5.0,1467896747
15347552,99476,32,3.5,1467896709
15347553,99476,36,3.0,1467896896
15347554,99476,47,4.5,1467896676


In [4]:
relevant_movies = set(
    user_ratings[
        user_ratings["rating"] >= 4.0
    ]["movieId"]
)

print("Relevant Movies:", len(relevant_movies))

Relevant Movies: 117


In [5]:
def precision_at_k(recommended, relevant, k=10):

    recommended = recommended[:k]

    hits = len(
        set(recommended).intersection(relevant)
    )

    return hits / k

In [7]:
from sklearn.metrics import ndcg_score

def ndcg_at_k(recommended, relevant, k=10):

    recommended = recommended[:k]

    y_true = [
        1 if movie in relevant else 0
        for movie in recommended
    ]

    y_score = list(range(k, 0, -1))

    return ndcg_score([y_true], [y_score])

In [8]:
cf_movies = cf["movieId"].tolist()

cf_precision = precision_at_k(
    cf_movies,
    relevant_movies
)

cf_ndcg = ndcg_at_k(
    cf_movies,
    relevant_movies
)

In [9]:
hybrid_movies = hybrid["movieId"].tolist()

hybrid_precision = precision_at_k(
    hybrid_movies,
    relevant_movies
)

hybrid_ndcg = ndcg_at_k(
    hybrid_movies,
    relevant_movies
)

In [10]:
ctr = cf.sort_values(
    "ctr_probability",
    ascending=False
)

ctr_movies = ctr["movieId"].head(10).tolist()

ctr_precision = precision_at_k(
    ctr_movies,
    relevant_movies
)

ctr_ndcg = ndcg_at_k(
    ctr_movies,
    relevant_movies
)

In [11]:
comparison = pd.DataFrame({

    "Model": [
        "Collaborative Filtering",
        "CTR Prediction",
        "Hybrid Recommendation"
    ],

    "Precision@10": [
        cf_precision,
        ctr_precision,
        hybrid_precision
    ],

    "NDCG@10": [
        cf_ndcg,
        ctr_ndcg,
        hybrid_ndcg
    ]
})

comparison

,Model,Precision@10,NDCG@10
0,Collaborative Filtering,0.3,0.424960
1,CTR Prediction,0.3,0.773746
2,Hybrid Recommendation,0.3,0.539107


In [12]:
comparison.to_csv(
    "../model_comparison.csv",
    index=False
)

print("Model comparison saved successfully!")

Model comparison saved successfully!
